# module-extra-repr — worked example 2: extra_repr printing a float and a bool

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-extra-repr`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`extra_repr` can summarize any configuration, including floats and boolean flags. The rule is the same as for tensors: print a compact human-readable summary, not the underlying data. A boolean derived from `is not None` is the idiomatic way to show presence-or-absence without leaking the object itself.

## Worked solution

We define `AffineScale`, a module that multiplies its input by a learnable scale and optionally adds a learnable shift. In `__init__` we call `super().__init__()`, store the float `init_scale` and a `learnable` flag, then create the `scale` Parameter. The `shift` Parameter is created only when `use_shift` is True, otherwise we set `self.shift = None`. In `extra_repr` we report `init_scale` as a formatted float and report shift presence as the boolean `self.shift is not None` rather than printing the shift tensor. Printing the instance shows the f-string nested inside `AffineScale(...)`, confirming our override feeds the standard repr machinery.

In [ ]:
import torch as t

class AffineScale(t.nn.Module):
    def __init__(self, init_scale, use_shift=True):
        super().__init__()
        self.init_scale = float(init_scale)
        self.scale = t.nn.Parameter(t.tensor(float(init_scale)))
        if use_shift:
            self.shift = t.nn.Parameter(t.zeros(()))
        else:
            self.shift = None

    def extra_repr(self):
        return (f'init_scale={self.init_scale:.3f}, '
                f'has_shift={self.shift is not None}')

    def forward(self, x):
        out = x * self.scale
        if self.shift is not None:
            out = out + self.shift
        return out

mod = AffineScale(2.5, use_shift=False)
print(repr(mod))